In [2]:
# Import Dash for web dashboard
from jupyter_dash import JupyterDash

# Import dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import database module
from CRUD_Python_Module import AnimalShelter

###########################
# Connect to Database
###########################

username = "aacuser"
password = "Password123!"

db = AnimalShelter(username, password)

# Get all animals
df = pd.DataFrame.from_records(db.read({}))

# Remove _id column
if '_id' in df.columns:
    df.drop(columns=['_id'], inplace=True)

###########################
# Dashboard Layout
###########################

app = JupyterDash(__name__)

# Load logo
image_filename = 'Grazioso_Salvare_Logo.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

app.layout = html.Div([
    # Logo and title
    html.Div([
        html.A([
            html.Img(
                src='data:image/png;base64,{}'.format(encoded_image.decode()),
                style={'height': '80px', 'margin': '10px'}
            )
        ], href='https://www.snhu.edu', target='_blank'),
        html.Center([
            html.H1('Grazioso Salvare Animal Shelter Dashboard'),
            html.P('Dashboard created by Jasmyne Fisher', style={'fontSize': '14px', 'color': '#666'})
        ])
    ]),
    
    html.Hr(),
    
    # Filter buttons
    html.Div([
        html.H3('Select Rescue Type:', style={'margin': '10px'}),
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': ' Water Rescue', 'value': 'Water'},
                {'label': ' Mountain or Wilderness Rescue', 'value': 'Mountain'},
                {'label': ' Disaster or Individual Tracking', 'value': 'Disaster'},
                {'label': ' Reset (Show All)', 'value': 'Reset'}
            ],
            value='Reset',
            labelStyle={'display': 'inline-block', 'margin': '10px'},
            style={'margin': '10px'}
        )
    ], style={'backgroundColor': '#f0f0f0', 'padding': '10px', 'borderRadius': '5px'}),
    
    html.Hr(),
    
    # Data table
    html.Div([
        html.H3('Available Dogs'),
        dash_table.DataTable(
            id='datatable-id',
            columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
            data=df.to_dict('records'),
            editable=False,
            filter_action="native",
            sort_action="native",
            sort_mode="multi",
            column_selectable=False,
            row_selectable="single",
            row_deletable=False,
            selected_columns=[],
            selected_rows=[0],
            page_action="native",
            page_current=0,
            page_size=10,
            style_cell={'textAlign': 'left', 'minWidth': '100px', 'width': '150px', 'maxWidth': '200px'},
            style_header={'backgroundColor': '#003366', 'color': 'white', 'fontWeight': 'bold'}
        )
    ]),
    
    html.Br(),
    html.Hr(),
    
    # Charts (side by side)
    html.Div(className='row',
         style={'display': 'flex'},
         children=[
             html.Div(id='graph-id', className='col s12 m6', style={'width': '50%'}),
             html.Div(id='map-id', className='col s12 m6', style={'width': '50%'})
         ])
])

###########################
# Callbacks (Make it Interactive)
###########################

@app.callback(
    Output('datatable-id', 'data'),
    [Input('filter-type', 'value')]
)
def update_dashboard(filter_type):
    # Filter data based on button clicked
    
    if filter_type == 'Water':
        query = {
            'animal_type': 'Dog',
            'breed': {'$regex': 'Labrador Retriever|Chesapeake Bay Retriever|Newfoundland', '$options': 'i'},
            'sex_upon_outcome': 'Intact Female',
            'age_upon_outcome_in_weeks': {'$gte': 26, '$lte': 156}
        }
    
    elif filter_type == 'Mountain':
        query = {
            'animal_type': 'Dog',
            'breed': {'$regex': 'German Shepherd|Alaskan Malamute|Old English Sheepdog|Siberian Husky|Rottweiler', '$options': 'i'},
            'sex_upon_outcome': 'Intact Male',
            'age_upon_outcome_in_weeks': {'$gte': 26, '$lte': 156}
        }
    
    elif filter_type == 'Disaster':
        query = {
            'animal_type': 'Dog',
            'breed': {'$regex': 'Doberman Pinscher|German Shepherd|Golden Retriever|Bloodhound|Rottweiler', '$options': 'i'},
            'sex_upon_outcome': 'Intact Male',
            'age_upon_outcome_in_weeks': {'$gte': 20, '$lte': 300}
        }
    
    else:  # Reset - show all
        query = {}
    
    # Get data from database
    df = pd.DataFrame.from_records(db.read(query))
    if '_id' in df.columns:
        df.drop(columns=['_id'], inplace=True)
    
    return df.to_dict('records')


@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")]
)
def update_graphs(viewData):
    # Create pie chart
    if viewData is None or len(viewData) == 0:
        return [html.Div('No data available')]
    
    dff = pd.DataFrame.from_dict(viewData)
    breed_counts = dff['breed'].value_counts().reset_index()
    breed_counts.columns = ['breed', 'count']
    
    return [
        dcc.Graph(
            figure=px.pie(breed_counts, values='count', names='breed', title='Dog Breeds Distribution', hole=0.3)
        )
    ]


@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    # Highlight selected columns
    return [{'if': {'column_id': i}, 'background_color': '#D2F3FF'} for i in selected_columns]


@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")]
)
def update_map(viewData, index):
    # Show selected dog on map
    if viewData is None or len(viewData) == 0:
        return [html.Div('No data available')]
    
    dff = pd.DataFrame.from_dict(viewData)
    
    # Get selected row
    if index is None or len(index) == 0:
        row = 0
    else:
        row = index[0]
    
    if row >= len(dff):
        row = 0
    
    # Get location
    try:
        lat = dff.iloc[row]['location_lat']
        lon = dff.iloc[row]['location_long']
        breed = dff.iloc[row]['breed']
        name = dff.iloc[row]['name'] if dff.iloc[row]['name'] else 'Unnamed'
    except:
        lat, lon = 30.75, -97.48
        breed = 'Unknown'
        name = 'Unknown'
    
    # Create map
    return [
        dl.Map(
            style={'width': '100%', 'height': '500px'},
            center=[30.75, -97.48],
            zoom=10,
            children=[
                dl.TileLayer(id="base-layer-id"),
                dl.Marker(
                    position=[lat, lon],
                    children=[
                        dl.Tooltip(breed),
                        dl.Popup([html.H4("Animal Name"), html.P(name), html.H4("Breed"), html.P(breed)])
                    ]
                )
            ]
        )
    ]


# Start the dashboard
app.run_server(mode='jupyterlab')